# Exploratory Analysis

First thing first let's look at the class balance of the original train and test split.

In [8]:
from lab1_plots import plot_labels
from lab1_datasets import get_gtsrb_full_train, get_gtsrb_test
import numpy as np
from torch.utils.data import Dataset
from tqdm import tqdm

def get_labels_from_dataset(dataset: Dataset):
    return [y for _, y in tqdm(dataset)]

def plot_and_relative_balance():
    train_labels = get_labels_from_dataset(get_gtsrb_full_train())
    test_labels = get_labels_from_dataset(get_gtsrb_test())
    plot_labels(train_labels, test_labels)

    train_labels = np.bincount(train_labels)
    test_labels = np.bincount(test_labels)

    max_class_train = max(train_labels)
    min_class_train = min(train_labels)

    max_class_test = max(test_labels)
    min_class_test = min(test_labels)

    print(f'Train: {max_class_train / min_class_train}')
    print(f'Test: {max_class_test / min_class_test}')

plot_and_relative_balance()

KeyboardInterrupt: 

GTSRB is unbalanced in both train and test set. The majority class occurs 10 times more than the minority class (12.5 in the test set).

The classifier could show over chance level accuracy just by predicting the most common class

In [ ]:
import pandas as pd
from lab1_datasets import get_gtsrb_full_train
from torch.utils.data import Dataset

def show_image_info(dataset: Dataset):
    data = []
    for x, y in dataset:
        data.append({'width': x.width, 'height': x.height, 'ratio': x.width / x.height})
    return pd.DataFrame(data)

df = show_image_info(get_gtsrb_full_train())
df.plot()

In [ ]:
df['ratio'].hist(bins=100, legend=True)

Just a small percent of images has an extreme ratio, however it would be better to not change it, to preserve the shape of visual features.

# Feature Learning
Here we extract features from a backbone, and store them before learning.

In [3]:
import torch

from lab1_datasets import get_gtsrb_full_train, get_gtsrb_test
from backbones import Resnet18, MobileNetV3Small, ConvNextTiny, DinoV2, Backbone
from torch.utils.data import DataLoader

@torch.inference_mode()
def feature_extraction(
        backbones: list[Backbone],
        batch_size: int,
        num_workers: int,
    ):
    
    for i, b in enumerate(backbones):
        backbone_name = b.name()
        b = b()
        for ds_name, ds_constructor in [('train', get_gtsrb_full_train), ('test', get_gtsrb_test)]:
            dl = DataLoader(
                dataset=ds_constructor(b.get_transform()),
                batch_size=batch_size,
                shuffle=False,
                num_workers=num_workers,
                pin_memory=True,
                persistent_workers=True
            )
            b.feature_extraction(
                loader=dl,
                x_file=f'features/gtrsb_{backbone_name}_{ds_name}.pt',
                y_file=f'labels/gtrsb_{ds_name}.pt' if i == 0 else None,
                progress_bar=True
            )
            del dl

        del b

my_backbones = [Resnet18, MobileNetV3Small, ConvNextTiny, DinoV2]



In [ ]:
from dataclasses import dataclass
from typing import Literal

from feature_learning import feature_learning


feature_datasets = [FeatureDataset(b.name()) for b in my_backbones]
feature_learning([], train_features, test_features, y_train='labels/gtsrb_full_train.pt', y_test='labels/gtsrb_test.pt', rng_seed=0, result_csv_path='lab1_backbone_evaluation.csv')

ValueError: Expected 2D array, got scalar array instead:
array=StandardScaler().
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
import pandas as pd
df = pd.read_csv('lab1_backbone_evaluation.csv')
max_ba = df['BA_Test'].idxmax()
max_ac = df['ACC_Test'].idxmax()
print('Best BA', df.loc[max_ba, 'BA_Test'])
print('Best ACC', df.loc[max_ac, 'ACC_Test'])

# Fine tuning

For choosing a backbone I asked GPT-6 to search for a github repo where someone finetuned a backbone on GTRSB.

This one showed good results: https://github.com/tajwarchy/traffic-sign-classifier

So we will start with efficientnet_b0 and do some hyperparameter tuning

# Retrieval

Pipeline:
- backbone fine tuned in the last exercise
- standardization of features
- actual retrieval